In [2]:
%pip install numpy matplotlib ipython ipykernel scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import scipy.stats as stats

import matplotlib.pyplot as plt


## Опис

Спостерігається вибірка $ \bar{X} = \left(X_1, ..., X_n\right) $ , де $\left\{X_i\right\}$ – незалежні однаково розподілені випадкові величини, які мають показниковий розподіл з параметром $\lambda$, тобто $F(u, \lambda) = P\left\{X_i < u\right\} = 1 - \exp{\left\{-\lambda u\right\}}, u \ge 0$.

Якщо $\left\{\omega_i\right\}$ – незалежні рівномірно розподілені на відрізку $ [0, 1] $ в.в., то

\begin{equation}
    X_i = F^{-1}(1-\omega_i ; \lambda) = -\frac{1}{\lambda} \ln{\omega_i}
\end{equation}

Перевірку статистичних гіпотез вести при рівні значимості $\gamma = 0.05$. Кожне з наступних чотирьох завдань виконувати для $n=1000$, $n=10 000$ та $n=100 000$.

Користуючись перетворенням $Y_i = F(X_i ; \lambda), i = 1,...,n$, перевіряти на рівномірність випадкові величини $\left\{Y_i\right\}$ ($\text{\underline{лише перші три завдання}}$).

In [4]:
def F(u: float, lambd: float) -> float:
    return 1 - np.exp(-lambd * u)

def F_s(omega: float, lambd: float) -> float:
    return -1/lambd * np.log(omega)

def generate_X(n: int, lambd: float):
    omega = np.random.random(n)
    X = F_s(omega, lambd)
    return X

In [5]:
gamma = 0.05

ns = np.array([10**3, 10**4, 10**5])

X = [generate_X(n, 1) for n in ns]
Y = [F(x, 1) for x in X]
# a)
X_H0a = X
Y_H0a = Y

# b)
X_H0b = [generate_X(n, 1.2) for n in ns]
Y_H0b = [F(x, 1) for x in X_H0b]


## Вимоги

$\gamma = 0.05$

$n = 10^3, 10^4, 10^5$

***
# Завдання 1

## Вимоги

за допомогою критерія Колмогорова перевірити гіпотези:
- a) $ H_0: X_i \sim F(u, 1) $, коли насправді $ X_i \sim F(u, 1) $;
- b) $ H_0: X_i \sim F(u, 1) $, коли насправді $ X_i \sim F(u, 1.2) $.

$Y_i = F(X_i ; \lambda), i = 1,...,n$

## Реалізація

Алгоритм перевірки гіпотези $H_0$.
1. Обираємо рівень значимості $\gamma$ та за таблицею для розподілу Колмогорова знаходимо $z_{\gamma}$ таке, що $K(z_{\gamma}) = 1 - \gamma$; $\left( K(z) = \sum\limits_{k=-\infty}^{\infty}{(-1)^k e^{-2 k^2 z^2}} \right)$.
2. За формулою $D_n(\bar{x}) = \max\limits_{1 \le k \le n}\left\{\max\left\{ \left(  F(x_{(k)}) - \frac{k-1}{n} \right), \left( \frac{k}{n} - F(x_{(k)}) \right) \right\}\right\}$ обчислюємо $D_n(\bar{x})$.
3. Якщо $\sqrt{n}D_n(\bar{x}) < z_{\gamma}$, то робимо висновок: статистичні дані не суперечать гіпотезі $H_0$. Якщо ж $\bar{x} \in Z_1$, то слід прийняти альтернативну гіпотезу $H_1$.

In [42]:
def Colmogorov(Ys, ns, text = ""):  #, gamma
    # z = 1.36
    z = stats.kstwobign.ppf(1 - gamma)
    for i, Y in enumerate(Ys):
        n = ns[i]

        Dn = -float("inf")
        y_sorted = np.sort(Y)
        for k, y in enumerate(y_sorted):
            Dn = np.max([Dn, np.max([y - (k)/(n), (k+1)/(n) - y])])
        # Dn, _ = stats.kstest(Y, 'uniform')

        crit = z/np.sqrt(n)
        compare = Dn < crit
        status = "[Accept H0]" if compare else "[Decline H0]"
        sign = "<" if compare else ">="

        print(f"{status:<12} | {Dn:6.4f} {sign:<2} {crit:6.4f} | {text} n = {n:6}")

Colmogorov(Y_H0a, ns, "a)")
Colmogorov(Y_H0b, ns, "b)")

[Accept H0]  | 0.0184 <  0.0429 | a) n =   1000
[Accept H0]  | 0.0088 <  0.0136 | a) n =  10000
[Accept H0]  | 0.0028 <  0.0043 | a) n = 100000
[Decline H0] | 0.0872 >= 0.0429 | b) n =   1000
[Decline H0] | 0.0674 >= 0.0136 | b) n =  10000
[Decline H0] | 0.0672 >= 0.0043 | b) n = 100000


***
# Завдання 2

## Вимоги

за допомогою критерія $\chi^2$ перевірити гіпотези:
- a) $ H_0: X_i \sim F(u, 1) $, коли насправді $ X_i \sim F(u, 1) $;
- b) $ H_0: X_i \sim F(u, 1) $, коли насправді $ X_i \sim F(u, 1.2) $.

$\text{\underline{Зауваження}}$. Кількість проміжків $r$ обирати з умови: $r = 30 \cdot \frac{n}{1000}$.

$Y_i = F(X_i ; \lambda), i = 1,...,n$

$r = 30 \cdot \frac{n}{1000}$

## Реалізація

Алгоритм перевірки гіпотези $H_0$ за критерієм $\chi^2$.
1. Обираємо $\gamma$ та за таблицею для $\chi^2$-розподілу знаходимо $z_{\gamma}$ таке, що $T_{r-1}(z_{\gamma}) = 1 - \gamma$.
2. Обчислюємо ймовірності $\left\{ p_i \right\}$ та частоти $\left\{ v_i \right\}$.
3. За формулою (5) $\Delta_n^{(r)} = \sum\limits_{i=1}^{r}{\frac{(v_i - n p_i)^2}{n p_i}} = \sum\limits_{i=1}^{r}{\frac{v_i^2}{n p_i}} - n$ обчислюємо $\Delta_n^{(r)}$.
3. Якщо $\Delta_n^{(r)} < z_{\gamma}$, то робимо висновок: статистичні дані не суперечать гіпотезі $H_0$.  
Якщо ж $\Delta_n^{(r)} \ge z_{\gamma}$, тобто $\bar{v} \in Z_1$, то слід прийняти альтернативну гіпотезу $H_1$.

$\text{\underline{Зауваження}}$: 
1) критерій $\chi^2$ можна використовувати на практиці, якщо $n \ge 50$ та $v_i \ge 5$ для будь-яких $i=1,...,r$;
2) критерій $\chi^2$ є спроможним, тобто, якщо ймовірності потрапляння у області $\left\{ U_i \right\}$ не дорівнюють $\left\{ p_i \right\}$ (гіпотеза $H_1$), то потужність критерію прямує до одиниці (або ймовірність похибки другого роду прямує до нуля).

In [39]:
rs = (30 * ns / 1000).astype(int)
def Chi2(Ys, ns, rs, text = ""):  #, gamma
    for i, Y in enumerate(Ys):
        z = stats.chi2.ppf(1 - gamma, df=rs[i] - 1)

        r = rs[i]
        bins = np.linspace(0, 1, r+1)
        v, _ = np.histogram(Y, bins=bins)
        p = np.ones(r)/r

        delta = 0
        n = ns[i]
        for j in range(r):
            delta += (v[j]**2)/(n * p[j])
        delta -= n

        compare = delta < z
        status = "[Accept H0]" if compare else "[Decline H0]"
        sign = "<" if compare else ">="

        print(f"{status:<12} | {delta:9.4f} {sign:<2} {z:9.4f} | {text} n = {n:6}")

Chi2(Y_H0a, ns, rs, "a)")
Chi2(Y_H0b, ns, rs, "b)")

[Accept H0]  |   26.2400 <    42.5570 | a) n =   1000
[Accept H0]  |  301.5800 <   340.3279 | a) n =  10000
[Accept H0]  | 2975.3000 <  3127.5154 | a) n = 100000
[Decline H0] |   57.9800 >=   42.5570 | b) n =   1000
[Decline H0] |  564.8000 >=  340.3279 | b) n =  10000
[Decline H0] | 5757.0200 >= 3127.5154 | b) n = 100000


***
# Завдання 3

## Вимоги

за допомогою критерія пустих ящиків (асимптотична теорема) перевірити гіпотези:
- a) $ H_0: X_i \sim F(u, 1) $, коли насправді $ X_i \sim F(u, 1) $;
- b) $ H_0: X_i \sim F(u, 1) $, коли насправді $ X_i \sim F(u, 1.2) $.

$\text{\underline{Зауваження}}$. Кількість проміжків $r$ обирати з умови: $\rho = 2$, тобто із співвідношення $\frac{n}{r} = \rho$ випливає, що $r=\frac{n}{2}$.

$Y_i = F(X_i ; \lambda), i = 1,...,n$

$ \rho = 2 $

$ r = \frac{n}{\rho} $

## Реалізація

In [ ]:
rho = 2
rs = ns/rho

***
# Завдання 4

## Вимоги

за допомогою критерія однорідності Смирнова перевірити гіпотези:
- a) $H_0: \bar{X}^{(1)} = \left( X_1^{(1)}, ..., X_n^{(1)}\right) \sim F(u; 1)$, $\bar{X}^{(2)} = \left( X_1^{(2)}, ..., X_m^{(2)}\right) \sim F(u; 1)$ (саме так ці вибірки і генерувались);
- b) $H_0: \bar{X}^{(1)} = \left( X_1^{(1)}, ..., X_n^{(1)}\right) \sim F(u; 1)$, $\bar{X}^{(2)} = \left( X_1^{(2)}, ..., X_m^{(2)}\right) \sim F(u; 1)$ (насправді: $\bar{X}^{(1)} \sim F(u; 1)$, $\bar{X}^{(2)} \sim F(u; 1.2)$).

$\text{\underline{Зауваження}}$. Обирати $m=\frac{n}{2}$.

$m = \frac{n}{2}$


## Реалізація

$\text{\underline{Алгоритм перевірки гіпотези}}$ $H_0$.
1. Обираємо рівень значимості $\gamma$ та за таблицею для розподілу Колмогорова знаходимо $z_{\gamma}$ таке, що $K(z_{\gamma}) = 1 - \gamma$.
2. Обчислюємо $D_{n, m}(\bar{x}, \bar{y}) = \max{\left(D_{n, m}^{+}, D_{n, m}^{-}\right)}$, де
,
 – це реалізації вибірок , а  – порядкові статистики, які відповідають вибірці .

In [ ]:
ms = ns/2

X = [[generate_X(n, 1) for n in ns], [generate_X(m, 1) for n, m in zip(ns, ms)]]
# a)
X_H0a4 = X

# b)
X_H0b4 = [X[0], [generate_X(m, 1.2) for m in ms]]